# KT4: every local step, in report order

each cell just runs my original script from its own folder,
the same as `cd <folder> && python3 <script>`, so outputs end up where they always were

before running:
- `data/` needs the three GEE rasters (already in the repo; to rebuild them, run `gee/00_gee_PortHills2017_stack.js` in the Code Editor)
- Python 3.12 with numpy, pandas, scipy, scikit-learn, matplotlib, rasterio, shap; R with terra, ggplot2, dplyr, tidyr
- section 0 needs a GEE login (`earthengine authenticate`), skip it if you only want the local results
- cells marked slow take a few minutes

In [ ]:
import subprocess, time
from pathlib import Path

ROOT = Path.cwd().parent          # the repository folder (this notebook lives in notebook/)

def run(folder, script):
    #run one script from its own folder and show what it prints
    cmd = ["Rscript", script] if script.endswith(".R") else ["python3", script]
    t0 = time.time()
    r = subprocess.run(cmd, cwd=ROOT / folder, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{folder}/{script} failed")
    print(f"✓ {folder}/{script}  ({time.time() - t0:.0f} s)")

## 0 GEE checks (Python API, needs login) · Section 3.1a, Figure 1

In [ ]:
run("gee/python", "01_check_preprocessing.py")     # scene clear %, gap fill, dNBR coverage

In [ ]:
run("gee/python", "04_fig_visual_inspection.py")   # Figure 1 visual inspection

## 1 Burn-severity response (must run first) · Table 3
builds the LCDB-corrected dNBR and the threshold T, everything after this reads them

In [ ]:
run("python/03_relationships", "00_build_regression_inputs.py")

## 2 Data exploration · Section 3.1 (Tables 1–2, Figures 2–3, Figure 12, reference shapefile)

In [ ]:
run("python/02_exploration", "05_reference_counts.py")          # Table 1
run("python/02_exploration", "06_export_reference_shapefile.py")  # shapefile for Section 7

In [ ]:
run("python/02_exploration", "02_outliers.R")                   # Table 2, Figure 12 (appendix box plots)
run("python/02_exploration", "03_normality_transforms.R")        # Figure 2
run("python/02_exploration", "07_fig_transformations.py")        # Figure 3

## 3 Relationships · Sections 3.2 and 4.1 (Figures 4–6, χ² test)

In [ ]:
run("python/03_relationships", "03_regression.py")
run("python/03_relationships", "04_figures.py")                  # Figure 6 regression

In [ ]:
run("python/03_relationships", "05_export_lcdb_severity.py")     # severity classes map
run("python/03_relationships", "06_fuel_severity_chisq.py")      # χ² test
run("python/03_relationships", "08_fig_fuel_severity.py")        # Figure 5

In [ ]:
run("python/03_relationships", "07_predictor_correlation.py")    # Figure 4

## 4 Classification · Sections 3.2, 3.3 and 4.2 (Tables 4, 5, 7; Figures 7–10)

In [ ]:
run("python/04_classification", "02_separability.py")           # Table 4 JM, Figure 7 box plots

In [ ]:
run("python/04_classification", "04_rf_classification.py")      # Table 5 predictor-set CV  (slow)

In [ ]:
run("python/04_classification", "06_shap_local.py")              # SHAP  (slow)
run("python/04_classification", "09_report_figures.py")         # Figure 8 SHAP

In [ ]:
run("python/04_classification", "07_cleared_pine_mask.py")
run("python/04_classification", "08_final_SSp.py")              # final map, Figure 10, Table 7
run("python/04_classification", "11_fig_confidence.py")          # Figure 9

## 5 Overlay (step 5 in the workflow) · Table 6
exports the fuel map as an ASCII grid (also what Map 1 and Map 2 were drawn from in ArcGIS Pro),
then counts the unburned share by fuel class and aspect

In [ ]:
run("python/04_classification", "10_export_for_arcgis.py")
run("python/03_relationships", "09_unburned_by_fuel_aspect.py")  # Table 6